# Project 1 AirBnb NYC listings Dataset, Cleaning and Analysis

# Imports and Loading the data

## 1. Imports

We import the following libraries:

In [ ]:
import numpy  as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as  plt
import re
from scipy import stats

## 2. Loading the Data

We load the raw CSV with low_memory=False to avoid dtype inference warnings on mixed-type columns. and then we use head() to get the first five values and finally we drop duplicates.

In [ ]:
df = pd.read_csv('Airbnb_Open_Data.csv', low_memory=False)
df.head()
df = df.drop_duplicates() # dropping rows that have identical copies

### Dropping Irrelevant Columns

The following columns are removed before any analysis:

Column with their reason for removal

**country / country code** -> All listings are in the US – zero variance                  
**lat / long** -> Not used in our questions                                               
**license** -> >99% missing; not relevant to our questions                                
**house_rules** -> Free-text field; not used in analysis                                  

Rows where every value is NaN are also dropped.

In [ ]:
print(df['country'])
print(df['country code'])
print(df['lat'])
print(df['long'])
print(df['license'])
print(df['house_rules'])

In [ ]:
df = df.drop(columns = ['country','country code', 'lat','long', 'license','house_rules']) #drop unnecessary columns for our analysis

df = df.dropna(how='all') # drop rows in which every value is NAN

## Initial Messiness Check

Before cleaning anything, we audit the raw dataset to understand:
- How many columns exist and their dtypes
- Which columns have missing values and how many
- Whether numeric columns were read as strings

This is our data before cleaning , we will compare it to the post-cleaning state later.

In [ ]:
df.info() # name is not essential, we'll only use it if necessary to describe hosts, same for IDS

### Visualising Missing Values (Before Cleaning)

The visualization below gives a view of where nulls are concentrated.

In [ ]:
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)

plt.figure(figsize=(12, 6))
plt.bar(null_counts.index, null_counts.values, color='purple')
plt.title('Null Count per Column (Before Cleaning)')
plt.xlabel('Columns')
plt.ylabel('Null Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 4. Data Cleaning & Preprocessing

We clean each column in this order:
1. **Categorical flags** (host_identity_verified, instant_bookable) — fill with safe defaults
2. **Numeric columns** (price, service fee, Construction year, etc.) — strip symbols, cast types, and we'll impute with median
3. **Review-related columns** (number of reviews, reviews per month, last review) — use domain logic (e.g. 0 reviews → 0 reviews/month)
4. **Geographic columns** (neighbourhood group, neighbourhood) — fix typos, fill from lookup dictionary
5. **Outlier clamping** — clip extreme values to realistic bounds.

# Preprocessing Start

### 4.1 filling host_identity_verified nulls

We first inspect the unique values to know the content of this column.

In [ ]:
df['host_identity_verified'].unique() # uncomfirmed, verified , nan

Missing values in host_identity_verified are filled with unconfirmed as this is the most cautious assumption

In [ ]:
df['host_identity_verified'] = df['host_identity_verified'].fillna('unconfirmed') # filling null values with unconfirmed just to be cautious

In [ ]:
df['host_identity_verified'].unique() #unconfirmed, verified only

### 4.2 Descriptive Statistics

We call df.describe() to get a high-level statistical summary 

In [ ]:
print(df['availability 365'].describe())
print(df['minimum nights'].describe())
print(df['Construction year'].describe())

In [ ]:
df.describe()
df['availability 365'].median()

### 4.3 Construction year

Construction year has a roughly symmetric distribution across the dataset's 2003–2021 range.

In [ ]:
# filling nulls with medians to pereserve trends
print("NaNs before:", df['Construction year'].isna().sum())
df['Construction year'] = df['Construction year'].fillna(df['Construction year'].median())
print("NaNs after:", df['Construction year'].isna().sum())
print("Median used:", df['Construction year'].median())
df.head()

In [ ]:
df['Construction year'] = np.floor(df['Construction year']) #flooring to the nearest whole number

### 4.4 price and service fee type Conversion

Both columns were read as strings because the raw data includes signs and commas We:
1. Strip the "$" and "," characters with a regex
2. Cast to float
3. Fill remaining NaN values with the median

minimum nights nulls are filled with 1

In [ ]:
# Before conversion
print("Price dtype before:", df['price'].dtype)
print("Service fee dtype before:", df['service fee'].dtype)
df['minimum nights'].isna().sum()

In [ ]:
# type conversion for service fee and price from string to number
df['service fee'] = df['service fee'].astype(str).str.replace(r'[\$,]', '', regex=True).str.strip().astype(float)
df['price'] = df['price'].astype(str).str.replace(r'[\$,]', '', regex=True).str.strip().astype(float)
df['service fee']=df['service fee'].fillna(df['service fee'].median()) # filling service fee with median
df['price']=df['price'].fillna(df['price'].median()) # filling price with median
df['minimum nights']=df['minimum nights'].fillna(1) # filling minimum night nulls with 1
print("Price dtype before:", df['price'].dtype)
print("Service fee dtype before:", df['service fee'].dtype)
df.isnull().sum()

### 4.5 Filling cancellation_policy nulls

We use the mode because it makes the assumption that a missing policy is most likely the one that most hosts choose.

In [ ]:
df['cancellation_policy'].isna().sum()

In [ ]:
df['cancellation_policy'].unique() # we'll just use mode

df['cancellation_policy'] = df['cancellation_policy'].fillna(df['cancellation_policy'].mode()[0])
df['cancellation_policy'].isna().sum()

### 4.6 Filling availability 365 nulls with median

In [ ]:
df['availability 365'].isna().sum()

In [ ]:
df['availability 365']=df['availability 365'].fillna(df['availability 365'].median()) # filling with median to preseve trends
df['availability 365'].isna().sum()

### 4.7 filling calculated host listings count nulls

we fill missing listing counts using each host's actual listing count derived from the data. This is more accurate than median imputation because the true value is recoverable from the dataset itself.

In [ ]:
df['calculated host listings count'].isna().sum()

In [ ]:
df['calculated host listings count'] = df['calculated host listings count'].fillna(df.groupby('host id')['id'].transform('nunique')) #filling null counts by counting the amount of listings per id
df['calculated host listings count'].isna().sum()

### 4.8 Clipping availability 365

Availability must be between 0 and 365 days and any value outside this range is clipped.

In [ ]:
print(df[df['availability 365'].lt(0) | df['availability 365'].gt(365)]['availability 365'])

In [ ]:
df['availability 365'] = df['availability 365'].clip(lower=0, upper=365) #clipping because there are only 365 days in a year and days cant be negative

In [ ]:
print(df[df['availability 365'].lt(0) | df['availability 365'].gt(365)]['availability 365'])

### 4.9 filling instant_bookable/ NAME/ host name nulls

Missing values are filled with False. Since if a host has not explicitly enabled instant booking, we assume it is off.

In [ ]:
df['instant_bookable'].isna().sum()

In [ ]:
df['instant_bookable']=df['instant_bookable'].fillna('False') #filling nulls with false

In [ ]:
df['instant_bookable'].isna().sum()

host name and name are filled with unknown

In [ ]:
print(df['NAME'].isna().sum())
print(df['host name'].isna().sum())

In [ ]:
df['NAME'] = df['NAME'].fillna('Unknown') # just fill with unknown

In [ ]:
df['host name'] = df['host name'].fillna('Unknown') #just fill with unknown

In [ ]:
print(df['NAME'].isna().sum())
print(df['host name'].isna().sum())

If a listing has 0 reviews, it logically should have 0 reviews per month too. This fixes inconsistent or missing data.

In [ ]:
df.loc[df['number of reviews'] == 0, 'reviews per month'] = 0

In [ ]:
df[df['last review'].isna()]

### 4.10 filling last review and number of reviews nulls

- If number of reviews == 0 → last review = No Reviews and reviews per month = 0
- If a last review date exists but number of reviews is null → at least 1 review must exist
- Remaining nulls in last review for listings with reviews are filled with the median date

In [ ]:
print(df['number of reviews'].isna().sum())
print(df['last review'].isna().sum())
print(df['reviews per month'].isna().sum())

In [ ]:
# Create a mask for rows that definitely have no reviews
no_reviews_mask = (df['number of reviews'] == 0)

# Fill only those rows with a placeholder
df.loc[no_reviews_mask, 'last review'] = df.loc[no_reviews_mask, 'last review'].fillna('No Reviews')
df.loc[(df['last review'].isnull()) & (df['number of reviews'].isnull()), 'last review'] = 'No Reviews'
df.loc[no_reviews_mask, 'reviews per month'] = df.loc[no_reviews_mask, 'reviews per month'].fillna(0)

In [ ]:
df.loc[(df['last review'].notna()) & (df['number of reviews'] == 0), 'number of reviews'] = 1 # if there is a last review date then there is atleast 1 review

In [ ]:
df['reviews per month'] = df['reviews per month'].fillna(0) #listings may be new and have not been reviewed so its okay if they have 0 reviews

In [ ]:
df.loc[(df['last review'].notna()) & (df['number of reviews'].isna()), 'number of reviews'] = 1 #if there is a last review date then the null is atleast 1

In [ ]:
df['last review'] = pd.to_datetime(df['last review'], errors='coerce')
median_date = df['last review'].median()
df.loc[(df['last review'].isnull()) & (df['number of reviews'] > 0), 'last review'] = median_date

In [ ]:
df['review rate number'] = df['review rate number'].fillna(df['review rate number'].median()) # Fill missing values in 'review rate number' with the median

In [ ]:
print(df['number of reviews'].isna().sum())
print(df['last review'].isna().sum())
print(df['reviews per month'].isna().sum())

### 4.11 Filling neighbourhood group nulls and missing data

Borough labels are missing for some rows, but the neighbourhood name is present. We build a neighbourhood → borough lookup dictionary from rows where both fields are non-null and use it to fill the missing borough values. Neighbourhoods with no match in the dictionary are left as Unknown.

In [ ]:
print(df['neighbourhood'].isna().sum())

In [ ]:
# create dictionary of Neighbourhood: Borough from the rows that aren't null
neigh_to_group = df.groupby('neighbourhood')['neighbourhood group'].first().to_dict()
# use that dictionary to fill the missing borough values
df['neighbourhood group'] = df['neighbourhood group'].fillna(df['neighbourhood'].map(neigh_to_group))
# Boroughs have multiple neighbourhoods so we fill it with unkown to avoid guessing
df['neighbourhood'] = df['neighbourhood'].fillna('Unknown')

In [ ]:
print(df['neighbourhood'].isna().sum())

### 4.12 Fixing Borough Name Typos

We discovered two misspelled borough names in the data: Brookln and Manhatan. These are corrected with a simple replacement map before any groupby operations.

In [ ]:
df['neighbourhood group'].unique()

In [ ]:
df['neighbourhood group'] = df['neighbourhood group'].str.strip().str.title()
# dealing with misspelled duplicates
df['neighbourhood group'] = df['neighbourhood group'].replace({
    'Brookln': 'Brooklyn',
    'Manhatan': 'Manhattan'
})
df['neighbourhood group'].unique()

### 4.13 Ordinal Encoding of cancellation_policy

We create a numeric encoding for the three-level ordinal policy: `flexible=0`, `moderate=1`, `strict=2`.

In [ ]:
df['cancellation_policy'].unique()

In [ ]:
cancel_order = {'flexible': 0, 'moderate': 1, 'strict': 2}
df['cancellation_policy_encoded'] = df['cancellation_policy'].map(cancel_order)
df['cancellation_policy_encoded'].unique()

### Descriptive statistics before outlier clipping

We run describe() again to spot any remaining anomalies before applying outlier clipping.

In [ ]:
df.describe()

### 4.14 Outlier Clipping

Extreme values in several columns are physically or logically impossible:

Column, their clip range and their justification

**minimum nights** -> [1, 30] >30 nights is an outlier; legally many cities cap short-term rentals
**number of reviews** -> [0, 99th percentile] Top 1% are extreme outliers                 
**reviews per month** -> [0, 99th percentile] Same rationale                              
**Construction year** -> [2003, 2021] Dataset's known valid range                         
**last review** -> [min, 2022-12-31] Reviews cannot be in the future relative to data collection

In [ ]:
# minimum nights
df['minimum nights'] = df['minimum nights'].clip(lower=1, upper=30)
# number of reviews  more than 99% is prolly not it
df['number of reviews'] = df['number of reviews'].clip(upper=df['number of reviews'].quantile(0.99))
# reviews per month same idea
df['reviews per month'] = df['reviews per month'].clip(upper=df['reviews per month'].quantile(0.99))
# Construction year clamp to valid
df['Construction year'] = df['Construction year'].clip(lower=2003,upper=2021)
#clamp last review
df['last review'] = df['last review'].clip(upper=pd.Timestamp('2022-12-31'))
df['review_year'] = df['last review'].dt.year

In [ ]:
df['instant_bookable'] = df['instant_bookable'].apply(lambda x: 1 if x == True or x == 'True' else 0)
df['instant_bookable'].unique()

### Descriptive statistics after outlier clipping

This is our data post cleaning. Compare with the pre-cleaning version above to confirm that outliers have been removed and all ranges are now logical.

In [ ]:
df.describe()

### Before vs After Cleaning — Null Summary

The cell below prints the number of remaining nulls per column. At this point the dataset should be fully populated (or only have intentional `NaN` values).

In [ ]:
df.isnull().sum()

### Visualising Missing Values (after Cleaning)

The visualization below gives a view of where nulls are concentrated after cleaning.

In [ ]:
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)

plt.figure(figsize=(12, 6))
plt.bar(null_counts.index, null_counts.values, color='purple')
plt.title('Null Count per Column (After Cleaning)')
plt.xlabel('Columns')
plt.ylabel('Null Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Final Shape & DTypes After Cleaning

df.info() confirms the final column count, dtypes, and that there are no unexpected object columns that should have been cast to numeric.

In [ ]:
df.info()

### Saving the Cleaned Dataset

We export the cleaned DataFrame to airbnb_cleaned.csv so that rest of the steps can load a consistent, pre-processed version without re-running all cleaning steps.

In [ ]:
df.to_csv("airbnb_cleaned.csv",index=False)


---
# Part 1 — Exploratory Data Analysis

We answer 12 business questions using the cleaned dataset. For each question we:
1. Generate a plot
2. Describe what we see in the visualisation
3. Answer the question
4. Extract deeper insights

---

#  Questions

## 1. How does service fee change across room types?

We first look at the distribution of service fees to see whether there is variance worth investigating before grouping by room type.

In [ ]:
df['service fee'].describe()  # doesnt seem like std is that high

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

room_fees = (df.groupby('room type')['service fee']
               .mean()
               .sort_values(ascending=False)
               .reset_index())

sns.barplot(
    data=room_fees,
    x='room type',
    y='service fee',
    ax=ax
)

ax.set_title('Average service fee by room type', fontsize=13, pad=12)
ax.set_xlabel('Room type')
ax.set_ylabel('Average service fee ($)')
sns.despine()
plt.tight_layout()
plt.show()

### Q1 Answer & Insights

**What we see in the plot:** The bar chart shows the average service fee for each room type. Hotel rooms have the highest average fee, followed by entire homes/apartments, then private rooms, and finally shared rooms.

**Answer:** Hotel rooms have the highest average service fee, but overall the differences between room types are quite small roughly within a 5–10 range.

**Insight:** The fact that service fees are very similar across all room types suggests that they are likely calculated as a fixed percentage of the listing price rather than being set separately for each room type.

## 2. How do listing prices relate to review score, cancellation policy, and availability?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
df['availability_group'] = pd.cut(
    df['availability 365'],
    bins=[0, 90, 270, 365],    labels=['Low', 'Medium', 'High']
) # bin so that its readable
# all plts -> price vs rev score, price vs plocy, plt price vs avail
# plts price vs review score
sns.barplot(data=df, x='review rate number', y='price',
            estimator='median', color='purple', ax=axes[0])
axes[0].set_title('Price vs review score')
axes[0].set_xlabel('Review rate number')
axes[0].set_ylabel('Median price in dollars')

# plt  price vs cancellation policy
sns.barplot(data=df, x='cancellation_policy', y='price',
            estimator='median', color='magenta', ax=axes[1])
axes[1].set_title('Price vs cancellation policy')
axes[1].set_xlabel('Cancellation policy')
axes[1].set_ylabel('Median price in dollars')

# price vs availability
sns.barplot(data=df, x='availability_group', y='price',
            estimator='median', color='fuchsia', ax=axes[2])
axes[2].set_title('Price vs availability')
axes[2].set_xlabel('Availability group')
axes[2].set_ylabel('Median price in dollars')

plt.suptitle('Listing price relationships', fontsize=14)
sns.despine()
plt.tight_layout()
plt.show()

### Q2 Answer & Insights

**What we see in the plots:**
- Price vs Review Score: Bars are nearly identical in height across all rating levels no upward or downward trend.
- Price vs Cancellation Policy:Flexible, moderate, and strict policies all cluster around the same median price.
- Price vs Availability: Low, medium, and high availability groups are priced almost identically.

**Answer:** Listing price does not meaningfully correlate with review score, cancellation policy, or availability.

**Insight:** This consistency suggests a possible issue with the dataset itself, since we would normally expect some variation (for example, higher-rated listings or more flexible policies often being priced higher). The lack of differences indicates low price variability across categories, which may point to data quality limitations or oversimplified pricing patterns in the dataset.

# 3. How does entire home price compare across neighborhood groups?

In [ ]:
entire_home = df[df['room type'] == 'Entire home/apt'] # filtered so that its ONLY entire home or entire apartment

fig, ax = plt.subplots(figsize=(10, 6))

order = (entire_home.groupby('neighbourhood group')['price'].mean().sort_values(ascending=False).index)

sns.boxplot(
    data=entire_home,
    x='neighbourhood group',
    y='price',
    order=order,
    ax=ax,color='purple'
)

ax.set_title('Entire home/apt price by neighbourhood group', fontsize=13, pad=20)
ax.set_xlabel('Neighbourhood group')
ax.set_ylabel('Price ')
sns.despine()
plt.tight_layout()
plt.show()

### Q3 Answer & Insights

**What we see in the plot:** The box plots for all five boroughs overlap a lot. The medians fall between 620–650 for each borough, and the spread IQR and whiskers look very similar as well.

**Answer:** Prices for entire homes/apartments are almost the same across all five NYC boroughs, meaning location does not significantly affect pricing for this room type.

**Insight:** This may suggest competitive pricing, where hosts align their prices with the general market rate across boroughs rather than adjusting based on location differences. It could also reflect the same low price variation seen in Q2.

## 4. Are there patterns in pricing across NYC boroughs and room types?

In [ ]:
# storing aggregations in variables to make the code more readble
mean_price_by_borough = df.groupby('neighbourhood group')['price'].mean()
mean_price_by_room = df.groupby('room type')['price'].mean()
overall_mean_price = df['price'].mean()

# showcasing the numbers
print("Overall mean price: " )
print(overall_mean_price)
print(f"\nMean Price by Borough:")
print(mean_price_by_borough)
print(f"\nMean Price by Room Type:")
print(mean_price_by_room)

# visualizing

# boxplot: price by room type
df.boxplot(column='price', by='room type', figsize=(12, 6))
plt.title('Price Distribution by Room Type')
plt.ylabel('Price (USD)')
plt.xlabel('Room Type')
plt.suptitle('')
plt.show()

# bar chart: average price by borough
plt.figure(figsize=(12, 6))
mean_price_by_borough.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Mean Price by Borough')
plt.xlabel('Borough')
plt.ylabel('Mean Price ($)')
plt.tight_layout()
plt.show()

### Q4 Answer & Insights

**What we see in the plots:** Both the box plot (price by room type) and the bar chart (average price by borough) show very tight distributions, with little difference between the groups.

**Answer:** There are no clear pricing patterns based on borough or room type. Prices appear very similar across the NYC Airbnb market in this dataset.

**Deeper insight:** The lack of a typical borough effect (for example, Manhattan usually being more expensive) is unusual. This supports the idea that the dataset’s price values may have been normalized or artificially generated. Because of this, any conclusions about pricing should be interpreted with caution.

# 5. How do single-listing and multi-listing hosts differ in their availability?

In [ ]:
# aggregation

df['host_type'] = df['calculated host listings count'].apply(lambda x: 'Single 1' if x == 1 else 'Multi 2+')

agg1 = df.groupby('host_type')['availability 365'].agg(['mean', 'median'])
print("Availability by Host Type:")
print(agg1)

# correlation
correlation = df['calculated host listings count'].corr(df['availability 365'])
print(f"\nCorrelation (listings vs. availability): {correlation}")

# visualizing

# violing plot: availabilty by host type
plt.figure(figsize=(10, 6))
sns.violinplot(data=df, x='host_type', y='availability 365')
plt.ylabel('Availability (days/year)')
plt.xlabel('Host Type')
plt.title('Availability Distribution: Single vs. Multi-Listing Hosts')
plt.show()

# bar plot: availability average by number of listings
df['listing_bucket'] = pd.cut(
    df['calculated host listings count'],
    bins=[0, 1, 3, 5, 10, float('inf')],
    labels=['1', '2-3', '4-5', '6-10', '10+']
)
bucket_avg = df.groupby('listing_bucket', observed=True)['availability 365'].mean().reset_index()

sns.barplot(data=bucket_avg, x='listing_bucket', y='availability 365')
ax2 = plt.gca()  # get the current axes
ax2.set_title('Avg Availability by Listing Count Bucket')
ax2.set_xlabel('Number of Listings (Bucketed)')
ax2.set_ylabel('Avg Availability (days/year)')

plt.show()

### Q5 Answer & Insights

**What we see in the plots:** The violin plot shows that hosts with multiple listings (2 or more) have a slightly higher concentration at the upper end of the availability range. The bar chart also shows a gradual increase in average availability as the number of listings increases.

**Answer:** Hosts with multiple listings tend to have higher year-round availability compared to those with only one listing.

**Deeper insight:** This aligns with a more professional hosting approach, where people managing several properties treat it as a business and keep them available most of the time. In contrast, single-listing hosts may block dates for personal use. The positive relationship between number of listings and availability supports this idea.

# 6. Are there trends in review activity across neighbourhoods that suggest shifts in guest preferences?

In [ ]:
# aggregatiion

# creating new dataframes so as to not affect the original
top_10_hoods = df.groupby('neighbourhood')['number of reviews'].sum().nlargest(10).index

agg1 = df.groupby(['review_year', 'neighbourhood']).agg({
    'number of reviews': 'sum',
    'id': 'count'
}).reset_index()
agg1['reviews_per_listing'] = (agg1['number of reviews'] / agg1['id']).round(2)

agg1_top10 = agg1[agg1['neighbourhood'].isin(top_10_hoods)]

agg2 = df.groupby(['review_year', 'room type']).agg({
    'number of reviews': 'sum',
    'id': 'count'
}).reset_index()
# divide each value by the year's total, multiply by 100 because percentage
agg2['pct_of_reviews'] = agg2.groupby('review_year')['number of reviews'].transform(lambda x: x/x.sum()*100).round(1)


agg3 = df.groupby('neighbourhood').agg({
    'number of reviews': 'sum',
    'reviews per month': 'mean',
    'id': 'count'
})
agg3['reviews_per_listing'] = (agg3['number of reviews'] / agg3['id']).round(2)
agg3 = agg3.sort_values('number of reviews', ascending=False)
print("Top Neighbourhoods by Review Activity:")
print(agg3.head())

# visualizing
# line plot: total reviews per year
agg1_pivot = agg1_top10.pivot(index='review_year', columns='neighbourhood', values='number of reviews')
agg1_pivot.plot(marker='o', linewidth=2, markersize=8)
plt.title('Review Activity Trends by Neighbourhood')
plt.xlabel('Year')
plt.ylabel('Total Reviews')
plt.legend()
plt.show()

# Stacked bar: percentage of reviews by neighbourhood per year
room_type_pivot = agg2.pivot(index='review_year', columns='room type', values='pct_of_reviews')
plt.figure(figsize=(15, 6))
room_type_pivot.plot(kind='bar', stacked=True, ax=plt.gca())
plt.title('Traveller Preference Shift: neighbourhood Share Over Years')
plt.xlabel('Year')
plt.ylabel('% of Total Reviews')
plt.legend(title='Room Type', loc='upper left')
plt.show()

### Q6 Answer & Insights

**What we see in the plots:**
- Line plot: Bedford-Stuyvesant and Williamsburg have the highest number of reviews across all years. There is a clear drop around 2020, likely due to COVID.
- Stacked bar: Reviews for entire homes/apartments have increased over time, while reviews for private rooms have decreased.

**Answer:** Guest preferences are shifting toward entire homes/apartments and away from private rooms. Bedford-Stuyvesant and Williamsburg remain the most reviewed and likely most popular boroughs throughout the period.

**Deeper insight:** The drop in 2020 is an important factor, as COVID-19 likely pushed guests to avoid shared spaces, increasing demand for entire homes. This shift may not fully reverse even after the pandemic.

# 7. How does the number of reviews affect listing price across different room types?



In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=df,
    x='number of reviews',
    y='price',
    hue='room type',
    alpha=0.6
)

plt.title('Impact of Number of Reviews on Price by Room Type', fontsize=14)
plt.xlabel('Number of Reviews')
plt.ylabel('Price')
plt.legend(title='Room Type')
plt.tight_layout()

plt.show()

### Q7 Answer & Insights

**What we see in the plot:** The scatter plot shows a dense cluster near the lower end (few reviews) with a wide range of prices, and no clear upward or downward trend as the number of reviews increases.

**Answer:** The number of reviews does not have a significant impact on listing price across room types.

**Deeper insight:** While having more reviews may indicate that a listing is more established, it does not necessarily lead to higher prices. Pricing is likely driven more by supply and demand within a specific area. This also suggests that newer listings (with fewer reviews) are not consistently cheaper.

# 8. How does availability relate to review scores and does this relationship differ by neighbourhood group?



In [ ]:
df['availability level'] = pd.cut(
    df['availability 365'],
    bins=[0, 120, 240, 365],
    labels=['Low', 'Medium', 'High']
)

plt.figure(figsize=(10,6))

sns.boxplot(
    data=df,
    x='neighbourhood group',
    y='review rate number',
    hue='availability level'
)

plt.title('Review Scores vs Availability across Neighbourhood Groups', fontsize=14)
plt.xlabel('Neighbourhood Group')
plt.ylabel('Review Score')
plt.xticks(rotation=30)

plt.legend(title='Availability Level', loc='upper right')

plt.tight_layout()
plt.show()

print("\nAverage review score by neighbourhood group:")
print(df.groupby('neighbourhood group')['review rate number'].mean())

### Q8 Answer & Insights

**What we see in the plot:** Across all boroughs and availability levels, the box plots cluster around the same review score range 3.5–4.5. Medians are nearly identical regardless of availability level.

**Answer:** Availability does not have a meaningful impact on review scores, and this pattern is consistent across all five boroughs.

**Deeper insight:** Review scores seem to depend more on factors like amenities, cleanliness, host responsiveness, and location quality rather than how often a listing is available. Listings with high availability are not necessarily rated higher.

# 9. How do prices vary between hosts with one listing and hosts with multiple listings across different boroughs?

In [ ]:
# create host type
df['host_type'] = df['calculated host listings count'].apply(
    lambda x: 'Single 1' if x == 1 else 'Multi 2+'
)

plt.figure(figsize=(12,6))

sns.boxplot(
    data=df,
    x='neighbourhood group',
    y='price',
    hue='host_type'
)

plt.title('Price Variation by Host Type Across Neighbourhood Groups', fontsize=14)
plt.xlabel('Neighbourhood Group')
plt.ylabel('Price')
plt.xticks(rotation=30)
plt.legend(title='Host Type')

plt.tight_layout()
plt.show()

print("\nAverage price by host type:")
print(df.groupby('host_type')['price'].mean())

### Q9 Answer & Insights

**What we see in the plot:** The box plots for single vs. multi-listing hosts overlap quite a bit. However, multi-listing hosts show a slightly higher median price and a wider spread in some boroughs.

**Answer:** Hosts with multiple listings tend to price their properties slightly higher than those with only one listing, and this trend appears across different boroughs.

**Deeper insight:** This could be because more experienced or professional hosts have better knowledge of pricing strategies and market trends, allowing them to set higher prices. They may also own properties in more desirable areas within each borough.

# 10. How does the price of listings evolve across neighbourhoods?


In [ ]:
price_by_neigh = df.groupby('neighbourhood group')['price'].mean().reset_index()
price_by_neigh.columns = ['neighbourhood group', 'mean_price']
price_by_neigh = price_by_neigh.sort_values('mean_price')

plt.figure(figsize=(10, 6))
plt.bar(price_by_neigh['neighbourhood group'],
        price_by_neigh['mean_price'],
        color='steelblue', edgecolor='navy')
plt.ylim(600, 650)
plt.title("Mean Price by Neighbourhood Group")
plt.xlabel("Neighbourhood Group")
plt.ylabel("Mean Price in ($)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(price_by_neigh[['neighbourhood group', 'mean_price']].to_string(index=False))

In [ ]:
price_by_neigh = df.groupby('neighbourhood group')['price'].mean().reset_index()
price_by_neigh.columns = ['neighbourhood group', 'mean_price']
price_by_neigh = price_by_neigh.sort_values('mean_price')

plt.figure(figsize=(10, 6))
bars = plt.barh(price_by_neigh['neighbourhood group'], price_by_neigh['mean_price'],
                color=plt.cm.coolwarm([i/len(price_by_neigh) for i in range(len(price_by_neigh))]))
plt.title("Mean Price by Neighbourhood Group")
plt.xlabel("Mean Price ($)")
plt.ylabel("Neighbourhood Group")
for bar, val in zip(bars, price_by_neigh['mean_price']):
    plt.text(bar.get_width() - 1, bar.get_y() + bar.get_height()/2,
             f'${val:.1f}', va='center', ha='right', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

### Q10 Answer & Insights

**What we see from the visualization:** The bar chart y-axis zoomed to 600–650 reveals that Queens has the highest mean price while Manhattan has the lowest a difference of only 8.

**Answer:** There is no meaningful price evolution across neighbourhoods; pricing is essentially uniform.

**Deeper insight:** The very narrow y-axis range amplifies visual differences that are trivial. In a real Airbnb dataset, Manhattan would be expected to command a premium of 20–40% over outer boroughs. The absence of this gradient again points to low price diversity in the dataset.

# 11. Are entire homes/apartments significantly more expensive than private rooms?


In [ ]:
room_df = df[df['room type'].isin(['Entire home/apt', 'Private room'])]
room_price = room_df.groupby('room type')['price'].mean().sort_values().reset_index()
room_price.columns = ['room type', 'mean_price']

plt.figure(figsize=(7, 6))
plt.bar(room_price['room type'], room_price['mean_price'],
        color=['steelblue', 'coral'], edgecolor='navy')
plt.title("Mean Price: Entire Home vs Private Room")
plt.xlabel("Room Type")
plt.ylabel("Average Price in ($)")
plt.tight_layout()
plt.show()

print(room_price.to_string(index=False))

In [ ]:
room_df = df[df['room type'].isin(['Entire home/apt', 'Private room'])]

plt.figure(figsize=(8, 6))
sns.violinplot(x='room type', y='price', data=room_df, palette=['steelblue', 'coral'])
plt.title("Price Distribution: Entire Home vs Private Room")
plt.xlabel("Room Type")
plt.ylabel("Price ($)")
plt.tight_layout()
plt.show()

### Q11 Answer & Insights

**What we see in the plot:** The two bars are nearly the same height entire homes and private rooms have almost identical mean prices.

**Answer:** Entire homes/apartments are not significantly more expensive than private rooms in this dataset, which contradicts typical Airbnb market behaviour.

**Deeper insight:** This is a strong signal of a data quality issue in the price column. In practice, entire homes are typically 2–3× the price of private rooms in NYC.

# 12. Which areas show the most variation in availability throughout the year?

In [ ]:
availability_var = df.groupby('neighbourhood group')['availability 365'].std().sort_values(ascending=False)

plt.figure(figsize=(9, 6))
sns.boxplot(x='neighbourhood group', y='availability 365', data=df, order=availability_var.index)
plt.title("Availability Variation Across Neighbourhoods")
plt.xlabel("Neighbourhood Group")
plt.ylabel("Availability (days per year)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(availability_var)

In [ ]:
availability_var = df.groupby('neighbourhood group')['availability 365'].std().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.violinplot(x='neighbourhood group', y='availability 365', data=df,
               order=availability_var.index, palette='muted')
plt.title("Availability Variation Across Neighbourhoods")
plt.xlabel("Neighbourhood Group")
plt.ylabel("Availability (days per year)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print(availability_var)

### Q12 Answer & Insights

**What we see in the plot:** All five boroughs have wide box plots with similar IQR and whisker lengths. The printed standard deviations range from 129 to 134 days.

**Answer:** All borough groups show similarly high and unpredictable availability no single area stands out as notably more stable.

**Deeper insight:** The near-equal variance across boroughs suggests that host behaviour is not borough-specific. Platform-wide factors like holidays, local events, regulatory changes are likely to drive availability patterns uniformly across NYC.